# Download dei dati dal mercato italiano

In [1]:
## SETUP: librerie e percorsi

import os
import time
import warnings
import numpy as np
import pandas as pd
import yfinance as yf

from tqdm import tqdm

# Setto i percorsi alle cartelle
BASE_DIR = ".."

DATA_DIR = os.path.join(BASE_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
MANUAL_DIR = os.path.join(DATA_DIR, "manual")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")

START_DATE = "2015-01-01" # periodo inizio
END_DATE = "2025-01-01"  # esclusivo, quindi include tutto il 2024

In [2]:
# riporto i dati relativi ai ticker, ovvero codici identificativi di aziende nella libreria yahoo finance

tickers_data = [
    ["A2A", "A2A.MI", "A2A", "Utilities"],
    ["AMP", "AMP.MI", "Amplifon", "Healthcare"],
    ["AZM", "AZM.MI", "Azimut", "Financial Services"],
    ["BAMI", "BAMI.MI", "Banco BPM", "Banking"],
    ["BMED", "BMED.MI", "Banca Mediolanum", "Financial Services"],
    ["BMPS", "BMPS.MI", "Banca Monte dei Paschi di Siena", "Banking"],
    ["BPE", "BPE.MI", "BPER Banca", "Banking"],
    ["BC", "BC.MI", "Brunello Cucinelli", "Consumer Goods"],
    ["BZU", "BZU.MI", "Buzzi", "Construction Materials"],
    ["CPR", "CPR.MI", "Campari", "Food Beverage"],
    ["DIA", "DIA.MI", "Diasorin", "Healthcare"],
    ["ENEL", "ENEL.MI", "Enel", "Utilities"],
    ["ENI", "ENI.MI", "ENI", "Energy"],
    ["ERG", "ERG.MI", "ERG", "Energy"],
    ["FBK", "FBK.MI", "FinecoBank", "Banking"],
    ["G", "G.MI", "Assicurazioni Generali", "Insurance"],
    ["HER", "HER.MI", "Hera", "Utilities"],
    ["IG", "IG.MI", "Italgas", "Utilities"],
    ["INW", "INW.MI", "Inwit", "Telecommunications"],
    ["IP", "IP.MI", "Interpump", "Industrials"],
    ["ISP", "ISP.MI", "Intesa Sanpaolo", "Banking"],
    ["LDO", "LDO.MI", "Leonardo", "Industrials"],
    ["MB", "MB.MI", "Mediobanca", "Banking"],
    ["MONC", "MONC.MI", "Moncler", "Consumer Goods"],
    ["NEXI", "NEXI.MI", "Nexi", "Financial Technology"],
    ["PIRC", "PIRC.MI", "Pirelli", "Consumer Goods"],
    ["PST", "PST.MI", "Poste Italiane", "Financial Services"],
    ["PRY", "PRY.MI", "Prysmian", "Industrials"],
    ["RACE", "RACE.MI", "Ferrari", "Automobiles"],
    ["REC", "REC.MI", "Recordati", "Healthcare"],
    ["SPM", "SPM.MI", "Saipem", "Energy"],
    ["SRG", "SRG.MI", "Snam", "Utilities"],
    ["STLAM", "STLAM.MI", "Stellantis", "Automobiles"],
    ["STM", "STMMI.MI", "STMicroelectronics", "Technology"],
    ["TIT", "TIT.MI", "Telecom Italia", "Telecommunications"],
    ["TEN", "TEN.MI", "Tenaris", "Energy"],
    ["TRN", "TRN.MI", "Terna", "Utilities"],
    ["UCG", "UCG.MI", "UniCredit", "Banking"],
    ["UNI", "UNI.MI", "Unipol", "Insurance"],
]

# imposto i nomi delle colonne del DataFrame dei ticker
tickers = pd.DataFrame(
    tickers_data,
    columns=["symbol", "yahoo_ticker", "name", "sector"]
)

tickers_path = os.path.join(MANUAL_DIR, "tickers_italia_moderni.csv") # percorso tickers
tickers.to_csv(tickers_path, index=False) # salvo il DataFrame in un csv senza indici

tickers # stampo tabella dei tickers

,symbol,yahoo_ticker,name,sector
0,A2A,A2A.MI,A2A,Utilities
1,AMP,AMP.MI,Amplifon,Healthcare
2,AZM,AZM.MI,Azimut,Financial Services
3,BAMI,BAMI.MI,Banco BPM,Banking
4,BMED,BMED.MI,Banca Mediolanum,Financial Services
5,BMPS,BMPS.MI,Banca Monte dei Paschi di Siena,Banking
6,BPE,BPE.MI,BPER Banca,Banking
7,BC,BC.MI,Brunello Cucinelli,Consumer Goods
8,BZU,BZU.MI,Buzzi,Construction Materials
9,CPR,CPR.MI,Campari,Food Beverage


In [3]:
def download_raw_market_data(yahoo_ticker, start=START_DATE, end=END_DATE): # yahoo_ticker è il ticker usato da Yahoo Finance
    """
    Scarica dati grezzi da Yahoo Finance:
    - Open, High, Low, Close (prezzo di apertura, prezzo max, prezzo min, prezzo di chiusura)
    - Volume (numero di azioni scambiate nella giornata)
    - Dividends (dividendo distribuito per azione, ovvero una porzione di utili distribuiti dalla società agli azionisti)
    - Stock Splits (eventuale frazionamento o raggruppamento azionario)

    Non usa Adjusted Close per le analisi (viene tenuto per confronto).
    """
    
    ticker = yf.Ticker(yahoo_ticker) # creo l'oggetto ticker relativo all'azienda
    
    # scarico i dati 
    df = ticker.history(
        start=start,
        end=end,
        auto_adjust=False, # necessario, altrimenti i prezzi verrebbero corretti automaticamente
        actions=True # per avere anche dividendi e stock splits
    )
    
    if df.empty:
        return pd.DataFrame()
    
    df = df.reset_index() # altrimenti non posso selezionare la data, poiché la data è l'indice stesso.
    
    if "Date" not in df.columns:
        df = df.rename(columns={df.columns[0]: "Date"}) # se la prima colonna non è Date, la rinomino io
        
    df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None) # trasformo i valori in date, rimuovendo i fusi orari
    
    required_cols = [
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "Adj Close", # lo lascio anche se non lo usiamo, magari torna utile
        "Volume",
        "Dividends",
        "Stock Splits"
    ]
    
    # se la colonna non c'è nei dati scaricati, creo la colonna nel mio e ci metto tutti NaN
    for col in required_cols:
        if col not in df.columns:
            df[col] = np.nan
    
    df = df[required_cols].copy() # creo un nuovo dataframe con solo le colonne che mi interessano
    df["YahooTicker"] = yahoo_ticker # per ogni riga aggiungo il ticker yahoo di provenienza
    
    return df

In [4]:
all_raw = []
failed_downloads = []

for _, row in tqdm(tickers.iterrows(), total=len(tickers)): # ciclo su tutti i ticker, riga per riga (indici non mi servono). In più, lo stampo a schermo
    symbol = row["symbol"]
    yahoo_ticker = row["yahoo_ticker"]
    
    try:
        df = download_raw_market_data(yahoo_ticker) # provo a scaricare i dati relativi al ticker
        
        if df.empty:
            failed_downloads.append((symbol, yahoo_ticker, "empty dataframe"))
            continue
            
        df["Symbol"] = symbol
        df["Name"] = row["name"]
        df["Sector"] = row["sector"]
        
        output_file = os.path.join(RAW_DIR, f"{symbol}_{yahoo_ticker}_raw.csv") # setto nome e path di output
        df.to_csv(output_file, index=False)
        
        all_raw.append(df) 
        
        time.sleep(0.3) # per evitare errori da sovraccarico di richieste
        
    except Exception as e:
        failed_downloads.append((symbol, yahoo_ticker, str(e)))

raw_prices = pd.concat(all_raw, ignore_index=True) # unisco i risultati

raw_prices_path = os.path.join(RAW_DIR, "italy_raw_prices_2015_2024.csv")
raw_prices.to_csv(raw_prices_path, index=False) # salvo nel file di output finale

failed_downloads_df = pd.DataFrame(
    failed_downloads,
    columns=["symbol", "yahoo_ticker", "error"]
)

failed_downloads_df.to_csv(
    os.path.join(RAW_DIR, "failed_downloads.csv"),
    index=False
)

print("Titoli scaricati:", raw_prices["Symbol"].nunique())
print("Download falliti:", len(failed_downloads_df))

failed_downloads_df # stampo eventuali download falliti

100%|██████████| 39/39 [00:24<00:00,  1.57it/s]


Titoli scaricati: 39
Download falliti: 0


,symbol,yahoo_ticker,error


In [5]:
def adjust_prices_like_paper(df_stock, symbol): # dataframe titolo, simbolo, eventuali correzioni
    """
    Corregge il prezzo Close grezzo seguendo la logica del paper (Non usa Adj Close):
    
    1. correzione per dividendi;
    2. correzione per stock split;
    3. correzione per fattori manuali aggiuntivi.
    """
    
    df = df_stock.copy() # nuovo dataframe su cui lavorare
    df = df.sort_values("Date").reset_index(drop=True) # ordiniamo in ordine cronologico
    
    df["P_raw"] = df["Close"].astype(float) # creo colonna del prezzo grezzo
    df["Volume"] = df["Volume"].astype(float) # creo colonna del volume
    
    # 1. Correzione per dividendi
    
    df["Dividends"] = df["Dividends"].fillna(0).astype(float) # rimpiazzo i Na con 0
    
    df["dividend_factor_daily"] = 1.0 # valore di default (nessuna correzione)
    
    mask_div = (
        (df["Dividends"] > 0) &
        (df["P_raw"] > 0) &
        (df["P_raw"].notna())
    ) # identifico le date con un booleano che mi indica la presenza di dividendi
    
    df.loc[mask_div, "dividend_factor_daily"] = (
        1 + df.loc[mask_div, "Dividends"] / df.loc[mask_div, "P_raw"]
    ) # se c'è il dividendo, correggo il prezzo come nel paper
    
    df["dividend_factor_cumulative"] = df["dividend_factor_daily"].cumprod() # calcolo la produttoria come nel paper
    
    df["P_after_dividends"] = (
        df["P_raw"] * df["dividend_factor_cumulative"]
    ) # salvo il valore finale
    
    # 2. Stock split da Yahoo
    
    df["Stock Splits"] = df["Stock Splits"].fillna(0).astype(float) 
    df["k_split"] = 1.0 # valore di default
    
    mask_split = df["Stock Splits"] > 0 # anche qui flaggo se c'è uno split 
    
    # Yahoo: 2.0 significa split 2:1.
    # Nel paper, se il prezzo si dimezza, k = 0.5.
    df.loc[mask_split, "k_split"] = 1 / df.loc[mask_split, "Stock Splits"]
    
    # 3. Applicazione correzioni    
    # Formula del paper: prodotto dei k_T per T > t.
    future_split_product_inclusive = df["k_split"].iloc[::-1].cumprod().iloc[::-1]
    df["split_factor_future"] = future_split_product_inclusive.shift(-1).fillna(1.0)
    
    df["P_adjusted_manual"] = (
        df["P_after_dividends"] * df["split_factor_future"]
    )
    
    return df

In [6]:
def compute_log_returns_with_gap_rule(df_stock, max_gap=7): # dataframe del titolo + max gap fra due osservazioni
    """
    Calcola log-rendimenti usando il prezzo corretto manualmente.
    Se il prezzo precedente manca, cerca fino a max_gap osservazioni precedenti (come nel paper).
    """
    
    df = df_stock.copy()
    df = df.sort_values("Date").reset_index(drop=True)
    
    price = df["P_adjusted_manual"].astype(float) # estraggo il prezzo
    
    previous_valid_price = price.ffill(limit=max_gap).shift(1) # se nei max_gap giorni successivi c'è NaN, lo riempio con l'ultimo prezzo disponibile. poi shifto di uno "verso il basso", così ho effettivamente i prezzi precedenti
    
    df["log_return"] = np.log(price) - np.log(previous_valid_price) # calcolo il log-rendimento
    
    invalid = (
        price.isna() |
        previous_valid_price.isna() |
        (price <= 0) |
        (previous_valid_price <= 0)
    ) # creo una flag per evidenziare i casi non validi 
    
    df.loc[invalid, "log_return"] = np.nan # se il caso non è valido metto nan
    
    return df

In [7]:
def compute_traded_money(df_stock):
    """
    Calcola il controvalore scambiato:
    traded_money = prezzo grezzo * volume.
    """
    
    df = df_stock.copy()
    df["traded_money"] = df["P_raw"] * df["Volume"] # per definizione di controvalore
    
    invalid = (
        df["P_raw"].isna() |
        df["Volume"].isna() |
        (df["P_raw"] <= 0) |
        (df["Volume"] < 0)
    ) # flaggo i casi non validi
    
    df.loc[invalid, "traded_money"] = np.nan 
    
    return df

In [8]:
'''
    Ora, titolo per titolo, applichiamo le funzioni precedenti
'''

processed_list = []

for symbol in tqdm(raw_prices["Symbol"].unique()): # ciclo su tutti i simboli
    df_stock = raw_prices[raw_prices["Symbol"] == symbol].copy() # prendo da raw_prices solo le righe del titolo corrente
    
    df_stock = adjust_prices_like_paper(
        df_stock=df_stock,
        symbol=symbol
    )
    
    df_stock = compute_log_returns_with_gap_rule(df_stock, max_gap=7)
    df_stock = compute_traded_money(df_stock)
    
    processed_list.append(df_stock)

processed = pd.concat(processed_list, ignore_index=True) # concateno i risultati

processed_path = os.path.join(
    PROCESSED_DIR,
    "italy_prices_returns_traded_money_2015_2024.csv"
) # percorso output

processed.to_csv(processed_path, index=False)

processed.head() # stampo le prime 5 righe

  0%|          | 0/39 [00:00<?, ?it/s]

100%|██████████| 39/39 [00:00<00:00, 45.72it/s]


,Date,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,YahooTicker,...,Sector,P_raw,dividend_factor_daily,dividend_factor_cumulative,P_after_dividends,k_split,split_factor_future,P_adjusted_manual,log_return,traded_money
0,2015-01-02,0.8390,0.8430,0.8245,0.8355,0.482592,7532922.0,0.0,0.0,A2A.MI,...,Utilities,0.8355,1.0,1.0,0.8355,1.0,1.0,0.8355,NaN,6.293756e+06
1,2015-01-05,0.8305,0.8445,0.8080,0.8080,0.466708,12324588.0,0.0,0.0,A2A.MI,...,Utilities,0.8080,1.0,1.0,0.8080,1.0,1.0,0.8080,-0.033468,9.958267e+06
2,2015-01-06,0.8140,0.8145,0.7945,0.7960,0.459777,11915319.0,0.0,0.0,A2A.MI,...,Utilities,0.7960,1.0,1.0,0.7960,1.0,1.0,0.7960,-0.014963,9.484594e+06
3,2015-01-07,0.8015,0.8085,0.7910,0.7915,0.457178,8763488.0,0.0,0.0,A2A.MI,...,Utilities,0.7915,1.0,1.0,0.7915,1.0,1.0,0.7915,-0.005669,6.936301e+06
4,2015-01-08,0.8020,0.8280,0.7930,0.8270,0.477683,12809885.0,0.0,0.0,A2A.MI,...,Utilities,0.8270,1.0,1.0,0.8270,1.0,1.0,0.8270,0.043875,1.059378e+07


In [9]:
# CONTROLLO QUALITA

quality = (
    processed
    .groupby(["Symbol", "YahooTicker", "Name", "Sector"]) # raggruppo per titolo
    .agg(
        first_date=("Date", "min"), # data minima titolo
        last_date=("Date", "max"), #data max titolo
        n_prices=("P_raw", "count"), # numero di prezzi grezzi presenti
        n_returns=("log_return", "count"), # numero di rendimenti logaritmici
        missing_returns=("log_return", lambda x: x.isna().mean()), # ritorni mancanti
        n_dividends=("Dividends", lambda x: (x > 0).sum()),
        n_splits=("Stock Splits", lambda x: (x > 0).sum()),
        mean_volume=("Volume", "mean"), # volume di azioni scambiate
        mean_traded_money=("traded_money", "mean") # controvolare medio giornaliero scambiato
    )
    .reset_index() # riporto il group by a colonne normali
    .sort_values("n_returns", ascending=False) 
)

quality_path = os.path.join(
    PROCESSED_DIR,
    "data_quality_summary_italy_2015_2024.xlsx"
)

quality.to_excel(quality_path, index=False)

quality

,Symbol,YahooTicker,Name,Sector,first_date,last_date,n_prices,n_returns,missing_returns,n_dividends,n_splits,mean_volume,mean_traded_money
0,A2A,A2A.MI,A2A,Utilities,2015-01-02,2024-12-30,2543,2542,0.000393,10,0,1.105128e+07,1.583782e+07
1,AMP,AMP.MI,Amplifon,Healthcare,2015-01-02,2024-12-30,2543,2542,0.000393,9,0,5.804275e+05,1.337158e+07
2,AZM,AZM.MI,Azimut,Financial Services,2015-01-02,2024-12-30,2543,2542,0.000393,11,3,1.185887e+06,2.139412e+07
3,BAMI,BAMI.MI,Banco BPM,Banking,2015-01-02,2024-12-30,2543,2542,0.000393,6,0,1.830369e+07,5.404445e+07
4,BC,BC.MI,Brunello Cucinelli,Consumer Goods,2015-01-02,2024-12-30,2543,2542,0.000393,8,0,1.019537e+05,4.497296e+06
5,BMED,BMED.MI,Banca Mediolanum,Financial Services,2015-01-02,2024-12-30,2543,2542,0.000393,19,0,1.429010e+06,1.063074e+07
6,BMPS,BMPS.MI,Banca Monte dei Paschi di Siena,Banking,2015-01-02,2024-12-30,2543,2542,0.000393,1,3,3.776353e+06,2.876787e+07
7,BPE,BPE.MI,BPER Banca,Banking,2015-01-02,2024-12-30,2543,2542,0.000393,10,0,1.112133e+07,3.100325e+07
8,BZU,BZU.MI,Buzzi,Construction Materials,2015-01-02,2024-12-30,2543,2542,0.000393,11,0,6.678047e+05,1.290943e+07
9,CPR,CPR.MI,Campari,Food Beverage,2015-01-02,2024-12-30,2543,2542,0.000393,10,1,2.628796e+06,1.928173e+07


In [10]:
# Nel mio caso filtro per titoli che sono disponibili almeno per l'80% dell'intevallo osservato

total_trading_days = processed["Date"].nunique() # numero di date diverse nel dataset 

MIN_OBSERVATIONS = int(total_trading_days * 0.80) # soglia minima

eligible_symbols = (
    quality
    .query("n_returns >= @MIN_OBSERVATIONS") # seleziono solo i titoli che superano la soglia minima
    ["Symbol"]
    .tolist()
)

print("Giorni totali nel dataset:", total_trading_days)
print("Soglia minima osservazioni:", MIN_OBSERVATIONS)
print("Titoli eleggibili:", len(eligible_symbols))

eligible_symbols

Giorni totali nel dataset: 2543
Soglia minima osservazioni: 2034
Titoli eleggibili: 37


['A2A',
 'AMP',
 'AZM',
 'BAMI',
 'BC',
 'BMED',
 'BMPS',
 'BPE',
 'BZU',
 'CPR',
 'DIA',
 'ENEL',
 'ENI',
 'ERG',
 'FBK',
 'G',
 'HER',
 'ISP',
 'IP',
 'LDO',
 'MB',
 'PRY',
 'MONC',
 'TIT',
 'TRN',
 'REC',
 'SPM',
 'TEN',
 'STM',
 'STLAM',
 'SRG',
 'UCG',
 'UNI',
 'INW',
 'PST',
 'RACE',
 'IG']

In [11]:
# creo la matrice dei log-ritorni

returns_matrix = (
    processed[processed["Symbol"].isin(eligible_symbols)] # prendiamo i titoli che superano la soglia minima
    .pivot(index="Date", columns="Symbol", values="log_return") # trasformo da formato lungo a formato matrice
    .sort_index() # ordiniamo la matrice in ordine cronologico
)

returns_matrix_path = os.path.join(
    PROCESSED_DIR,
    "returns_matrix_italy_2015_2024.csv"
)

returns_matrix.to_csv(returns_matrix_path)

missing_by_symbol = returns_matrix.isna().mean().sort_values(ascending=False) # percentuale di valori mancanti per ogni titolo

display(returns_matrix.head())
display(missing_by_symbol)

Symbol,A2A,AMP,AZM,BAMI,BC,BMED,BMPS,BPE,BZU,CPR,...,REC,SPM,SRG,STLAM,STM,TEN,TIT,TRN,UCG,UNI
Date,,,,,,,,,,,,,,,,,,,,,
2015-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-05,-0.033468,0.025190,-0.011491,-0.044165,-0.013386,-0.026467,0.002282,-0.050539,-0.051795,-0.023059,...,0.014443,-0.049018,-0.033935,-0.021945,-0.016998,-0.055409,-0.030130,-0.033110,-0.068533,-0.043060
2015-01-06,-0.014963,0.000000,-0.020573,-0.048334,-0.041267,-0.005764,-0.039528,-0.038429,-0.022313,0.023059,...,-0.011385,-0.007203,-0.004579,0.015724,-0.009020,0.012674,0.008785,-0.006645,-0.013889,-0.020203
2015-01-07,-0.005669,0.026511,0.001123,-0.003734,0.023707,-0.010654,-0.001510,-0.022784,0.030305,0.009862,...,-0.008432,-0.025009,0.000510,0.008802,0.020384,-0.016935,-0.007609,-0.004454,-0.009838,-0.022183
2015-01-08,0.043875,-0.054972,0.019451,0.050037,0.001142,0.026899,0.116847,0.048885,0.021654,0.040390,...,0.012242,0.020783,0.023673,0.026953,0.038786,0.021125,0.027812,0.011099,0.047098,0.024731


Symbol
IG       0.186001
RACE     0.100669
PST      0.082580
INW      0.046795
A2A      0.000393
BC       0.000393
BAMI     0.000393
AZM      0.000393
AMP      0.000393
CPR      0.000393
DIA      0.000393
BMED     0.000393
BMPS     0.000393
ENI      0.000393
ENEL     0.000393
ERG      0.000393
FBK      0.000393
HER      0.000393
G        0.000393
BPE      0.000393
BZU      0.000393
ISP      0.000393
IP       0.000393
LDO      0.000393
MB       0.000393
PRY      0.000393
MONC     0.000393
REC      0.000393
SPM      0.000393
SRG      0.000393
STLAM    0.000393
STM      0.000393
TEN      0.000393
TIT      0.000393
TRN      0.000393
UCG      0.000393
UNI      0.000393
dtype: float64

In [12]:
# costruzione della matrice del controvalore scambiato

traded_money_matrix = (
    processed[processed["Symbol"].isin(eligible_symbols)] # seleziono titoli validi
    .pivot(index="Date", columns="Symbol", values="traded_money") # trasformo da formato lungo a formato matrice
    .sort_index()  # ordine cronologico
)

traded_money_matrix_path = os.path.join(
    PROCESSED_DIR,
    "traded_money_matrix_italy_2015_2024.csv"
)

traded_money_matrix.to_csv(traded_money_matrix_path)

traded_money_matrix.head()

Symbol,A2A,AMP,AZM,BAMI,BC,BMED,BMPS,BPE,BZU,CPR,...,REC,SPM,SRG,STLAM,STM,TEN,TIT,TRN,UCG,UNI
Date,,,,,,,,,,,,,,,,,,,,,
2015-01-02,6.293756e+06,6.471185e+05,1.292855e+07,3.676064e+07,1.934960e+06,7.937882e+06,2.320942e+07,1.877152e+07,7.354599e+06,6.305281e+06,...,1.593398e+06,3.175172e+07,1.700657e+07,8.241062e+07,2.106332e+07,3.438689e+07,4.453566e+07,2.577020e+07,2.679726e+08,1.523447e+06
2015-01-05,9.958267e+06,3.135082e+06,1.716829e+07,6.414669e+07,6.724700e+05,1.435648e+07,4.190188e+07,2.547414e+07,1.222882e+07,1.200439e+07,...,1.558319e+06,4.726768e+07,3.831496e+07,1.464600e+08,3.283805e+07,3.945277e+07,9.314700e+07,3.827707e+07,4.395347e+08,3.172864e+06
2015-01-06,9.484594e+06,1.199623e+06,1.655487e+07,5.659573e+07,8.984213e+05,1.222118e+07,2.250639e+07,3.259661e+07,1.667211e+07,1.082113e+07,...,1.862912e+06,4.365966e+07,2.959128e+07,1.517106e+08,2.679278e+07,3.825157e+07,6.736141e+07,3.533710e+07,4.138378e+08,5.032390e+06
2015-01-07,6.936301e+06,2.446382e+06,1.791220e+07,5.749248e+07,1.304345e+06,1.143915e+07,3.140171e+07,2.633209e+07,2.480936e+07,1.080855e+07,...,1.841514e+06,4.726631e+07,4.929094e+07,1.971928e+08,3.659174e+07,4.378267e+07,6.782117e+07,4.252612e+07,4.760391e+08,5.581380e+06
2015-01-08,1.059378e+07,5.241797e+06,1.512813e+07,5.202286e+07,5.244262e+05,1.369901e+07,8.813978e+07,2.941898e+07,1.368625e+07,2.061601e+07,...,1.068174e+06,4.534588e+07,5.223721e+07,1.659763e+08,3.678609e+07,3.110918e+07,1.035975e+08,3.690952e+07,5.197502e+08,7.052185e+06


In [13]:
# COSTRUISCO BENCHMARK PER CAPITALIZZAZIONE

def download_shares_outstanding(yahoo_ticker, start=START_DATE, end=END_DATE):
    """
    Prova a scaricare il numero storico di azioni in circolazione da yfinance.
    """
    
    ticker = yf.Ticker(yahoo_ticker) # creo oggetto ticker
    
    try:
        shares = ticker.get_shares_full(start=start, end=end) # ottengo totale azioni in circolazione
        
        if shares is None or len(shares) == 0:
            return pd.DataFrame()
        
        shares = shares.reset_index() # trasformo data da indice a colonna normale
        shares.columns = ["Date", "shares_outstanding"] # di conseguenza, setto nomi colonne
        shares["Date"] = pd.to_datetime(shares["Date"]).dt.tz_localize(None) # converto in formato data pandas senza timezone
        
        return shares
    
    except Exception:
        return pd.DataFrame()

In [14]:
shares_all = []
failed_shares = []

for _, row in tqdm(tickers.iterrows(), total=len(tickers)): # iteriamo sui tickers msotrando l'avanzamento
    symbol = row["symbol"]
    yahoo_ticker = row["yahoo_ticker"]
    
    shares = download_shares_outstanding(yahoo_ticker)
    
    if shares.empty:
        failed_shares.append((symbol, yahoo_ticker)) # se non è disponibile vado avanti
        continue
    
    shares["Symbol"] = symbol
    shares["YahooTicker"] = yahoo_ticker
    
    shares_all.append(shares)
    
    time.sleep(0.3) # per evitare problemi

if len(shares_all) > 0:
    shares_outstanding = pd.concat(shares_all, ignore_index=True) # concateno, se ci sono
else:
    shares_outstanding = pd.DataFrame(
        columns=["Date", "shares_outstanding", "Symbol", "YahooTicker"]
    )

shares_outstanding_path = os.path.join(
    RAW_DIR,
    "shares_outstanding_italy_2015_2024.csv"
)

shares_outstanding.to_csv(shares_outstanding_path, index=False)

failed_shares_df = pd.DataFrame(
    failed_shares,
    columns=["symbol", "yahoo_ticker"]
)

failed_shares_df.to_csv(
    os.path.join(RAW_DIR, "failed_shares_outstanding.csv"),
    index=False
)

print("Titoli con shares outstanding:", shares_outstanding["Symbol"].nunique())
print("Titoli senza shares outstanding:", len(failed_shares_df))

failed_shares_df

  0%|          | 0/39 [00:00<?, ?it/s]

100%|██████████| 39/39 [00:18<00:00,  2.09it/s]

Titoli con shares outstanding: 39
Titoli senza shares outstanding: 0


,symbol,yahoo_ticker


In [15]:
def add_market_cap_from_shares(processed, shares_outstanding):
    """
    Aggiunge shares_outstanding e market_cap al dataset.
    Usa merge_asof per portare in ogni data l'ultimo numero di azioni disponibile.
    """
    ## copio e riordino i due dataframe 

    df = processed.copy()
    df = df.sort_values(["Symbol", "Date"])
    
    if shares_outstanding.empty:
        df["shares_outstanding"] = np.nan
        df["market_cap"] = np.nan
        return df
    
    shares = shares_outstanding.copy()
    shares = shares.sort_values(["Symbol", "Date"])
    
    result = []
    
    for symbol in df["Symbol"].unique(): # per ogni azienda
        df_s = df[df["Symbol"] == symbol].copy() # seleziono il sotto-dataframe relativo all'azienda con prezzi, rendimenti e denaro scambiato
        sh_s = shares[shares["Symbol"] == symbol].copy() # seleziono il sotto-dataframe per le azioni in circolazione
        
        if sh_s.empty:
            df_s["shares_outstanding"] = np.nan
            df_s["market_cap"] = np.nan
        else:
            df_s = pd.merge_asof(
                df_s.sort_values("Date"),
                sh_s[["Date", "shares_outstanding"]].sort_values("Date"),
                on="Date",
                direction="backward"
            ) # uniamo i due dataframe, in ordine cronologico, scegliendo per ogni data l'ultimo valore di shares_oustanding disponibile (backwards)
            
            df_s["market_cap"] = df_s["P_raw"] * df_s["shares_outstanding"] # calcolo della capitalizzazione di mercato
        
        result.append(df_s)
    
    return pd.concat(result, ignore_index=True)

In [16]:
# aggiungiamo quindi la capitalizzazione di mercato

processed_full = add_market_cap_from_shares(
    processed=processed,
    shares_outstanding=shares_outstanding
)

processed_full_path = os.path.join(
    PROCESSED_DIR,
    "italy_full_dataset_2015_2024.csv"
)

processed_full.to_csv(processed_full_path, index=False)

processed_full.head()

,Date,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,YahooTicker,...,dividend_factor_daily,dividend_factor_cumulative,P_after_dividends,k_split,split_factor_future,P_adjusted_manual,log_return,traded_money,shares_outstanding,market_cap
0,2015-01-02,0.8390,0.8430,0.8245,0.8355,0.482592,7532922.0,0.0,0.0,A2A.MI,...,1.0,1.0,0.8355,1.0,1.0,0.8355,NaN,6.293756e+06,NaN,NaN
1,2015-01-05,0.8305,0.8445,0.8080,0.8080,0.466708,12324588.0,0.0,0.0,A2A.MI,...,1.0,1.0,0.8080,1.0,1.0,0.8080,-0.033468,9.958267e+06,NaN,NaN
2,2015-01-06,0.8140,0.8145,0.7945,0.7960,0.459777,11915319.0,0.0,0.0,A2A.MI,...,1.0,1.0,0.7960,1.0,1.0,0.7960,-0.014963,9.484594e+06,NaN,NaN
3,2015-01-07,0.8015,0.8085,0.7910,0.7915,0.457178,8763488.0,0.0,0.0,A2A.MI,...,1.0,1.0,0.7915,1.0,1.0,0.7915,-0.005669,6.936301e+06,NaN,NaN
4,2015-01-08,0.8020,0.8280,0.7930,0.8270,0.477683,12809885.0,0.0,0.0,A2A.MI,...,1.0,1.0,0.8270,1.0,1.0,0.8270,0.043875,1.059378e+07,NaN,NaN


In [17]:
# costruzione matrice di capitalizzazione di mercato

market_cap_matrix = (
    processed_full[processed_full["Symbol"].isin(eligible_symbols)] # per titoli validi
    .pivot(index="Date", columns="Symbol", values="market_cap")
    .sort_index()
)

market_cap_matrix_path = os.path.join(
    PROCESSED_DIR,
    "market_cap_matrix_italy_2015_2024.csv"
)

market_cap_matrix.to_csv(market_cap_matrix_path)

market_cap_availability = market_cap_matrix.notna().mean().sort_values()

display(market_cap_matrix.head())
display(market_cap_availability)

Symbol,A2A,AMP,AZM,BAMI,BC,BMED,BMPS,BPE,BZU,CPR,...,REC,SPM,SRG,STLAM,STM,TEN,TIT,TRN,UCG,UNI
Date,,,,,,,,,,,,,,,,,,,,,
2015-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Symbol
STM      0.178923
STLAM    0.187180
BC       0.611876
TEN      0.611876
BZU      0.612269
RACE     0.766418
ERG      0.774282
BAMI     0.797877
IG       0.814392
A2A      0.863547
MONC     0.872591
BMED     0.900511
TIT      0.912308
TRN      0.913488
FBK      0.913881
PRY      0.913881
IP       0.913881
AZM      0.915061
LDO      0.915847
PST      0.917814
AMP      0.917814
MB       0.923319
UCG      0.931184
HER      0.935116
ENEL     0.940228
UNI      0.940228
CPR      0.941408
G        0.942981
ISP      0.942981
BMPS     0.948879
INW      0.953598
SRG      0.964215
DIA      0.976013
BPE      0.976799
REC      0.982304
SPM      0.990956
ENI      1.000000
dtype: float64

In [18]:
returns_final = returns_matrix.copy()

# imposto soglia massima di valori di ritorni mancanti
max_missing_per_day = int(0.10 * returns_final.shape[1])

returns_final = returns_final[
    returns_final.isna().sum(axis=1) <= max_missing_per_day # applichiamo il filtro
]

returns_final_path = os.path.join(
    PROCESSED_DIR,
    "returns_final_for_mst_italy_2015_2024.csv"
)

returns_final.to_csv(returns_final_path)

print("Dimensioni matrice rendimenti:", returns_final.shape)

returns_final.head()

Dimensioni matrice rendimenti: (2424, 37)


Symbol,A2A,AMP,AZM,BAMI,BC,BMED,BMPS,BPE,BZU,CPR,...,REC,SPM,SRG,STLAM,STM,TEN,TIT,TRN,UCG,UNI
Date,,,,,,,,,,,,,,,,,,,,,
2015-06-23,-0.006231,0.023906,0.028721,-0.007072,0.004052,0.024470,0.007183,0.017284,-0.009139,0.011494,...,0.016719,0.024581,0.006728,0.012200,0.028021,0.011986,0.005895,0.006770,-0.010050,0.009653
2015-06-24,-0.011675,-0.001433,0.014243,-0.001291,-0.002893,-0.010873,-0.020662,-0.001838,-0.019313,0.005698,...,0.006713,0.013780,0.000894,-0.005005,-0.015915,0.007123,-0.010975,0.000482,-0.011723,-0.006132
2015-06-25,-0.006343,0.008565,-0.002609,0.020461,0.005777,0.013415,0.012448,0.027813,0.001559,-0.001421,...,-0.002061,-0.015764,-0.000447,-0.012261,0.018544,-0.019913,0.000000,-0.000964,0.016375,-0.013714
2015-06-26,0.032203,-0.004988,0.012238,-0.002535,-0.011005,0.004432,0.016360,0.008905,0.015456,0.002841,...,0.000516,-0.016016,0.000447,0.017980,0.014333,0.004014,0.004235,0.002408,-0.000774,0.035445
2015-06-29,-0.054263,-0.015114,-0.041390,-0.069628,-0.009362,-0.075421,-0.108069,-0.067221,-0.059236,-0.025136,...,-0.030353,-0.021934,-0.032686,-0.066298,-0.037566,-0.024332,-0.048494,-0.034250,-0.073870,-0.050253


In [19]:
# aziende effettivamente utilizzate

company_info = (
    tickers[tickers["symbol"].isin(eligible_symbols)]
    .copy()
    .sort_values("symbol")
)

company_info_path = os.path.join(
    PROCESSED_DIR,
    "company_info_italy_2015_2024.csv"
)

company_info.to_csv(company_info_path, index=False)

company_info

,symbol,yahoo_ticker,name,sector
0,A2A,A2A.MI,A2A,Utilities
1,AMP,AMP.MI,Amplifon,Healthcare
2,AZM,AZM.MI,Azimut,Financial Services
3,BAMI,BAMI.MI,Banco BPM,Banking
7,BC,BC.MI,Brunello Cucinelli,Consumer Goods
4,BMED,BMED.MI,Banca Mediolanum,Financial Services
5,BMPS,BMPS.MI,Banca Monte dei Paschi di Siena,Banking
6,BPE,BPE.MI,BPER Banca,Banking
8,BZU,BZU.MI,Buzzi,Construction Materials
9,CPR,CPR.MI,Campari,Food Beverage
